In [2]:
# CONFIGURATION
# Modifie le symbole ici pour analyser une autre paire
# Exemples de paires majeures :
#   EURUSD   EUR/USD
#   USDJPY   USD/JPY
#   GBPUSD   GBP/USD
#   USDCHF   USD/CHF
#   AUDUSD   AUD/USD
#   USDCAD   USD/CAD
#   NZDUSD   NZD/USD
SYMBOL = "USDJPY"
FULL_NAME = "USD/JPY"

# Timeframes
TF_DAILY  = {"interval": "1d", "period": "1mo"}
TF_H1     = {"interval": "1h", "period": "1mo"}
TF_INTRADAY = {"interval": "5m", "period": "1d"}

In [3]:
import pandas as pd
import numpy as np
import ta
from datetime import datetime

from lib.data import get_cached_data
from lib.analysis import (
    detect_candlestick_patterns,
    find_support_resistance,
    analyze_market_structure,
    build_price_action_signal,
)
from lib.ai import analyze_with_ai

In [4]:
# Donnees higher timeframe
print("=" * 60)
print(f"DAILY TIMEFRAME - {{SYMBOL}}")
print("=" * 60)
df_daily = get_cached_data(SYMBOL + "=X", TF_DAILY["interval"], TF_DAILY["period"])

print("\n" + "=" * 60)
print(f"1H TIMEFRAME - {{SYMBOL}}")
print("=" * 60)
df_1h = get_cached_data(SYMBOL + "=X", TF_H1["interval"], TF_H1["period"])

DAILY TIMEFRAME - {SYMBOL}
✓ Using cached data (age: 24m 46s)

1H TIMEFRAME - {SYMBOL}
✓ Using cached data (age: 24m 46s)


In [5]:
# Analyse daily
daily_levels = find_support_resistance(df_daily)
daily_structure = analyze_market_structure(df_daily)

# Indicateurs techniques daily
df_daily["rsi"] = ta.momentum.RSIIndicator(close=df_daily["Close"], window=14).rsi()
macd = ta.trend.MACD(close=df_daily["Close"])
df_daily["macd"] = macd.macd()
df_daily["macd_signal"] = macd.macd_signal()
df_daily["ma20"] = df_daily["Close"].rolling(20).mean()

last_d = df_daily.iloc[-1]
daily_rsi = round(float(last_d["rsi"]), 1)
daily_macd = "bullish" if last_d["macd"] > last_d["macd_signal"] else "bearish"
daily_trend = "uptrend" if float(last_d["Close"]) > float(last_d["ma20"]) else "downtrend"

print(f"RSI:      {daily_rsi}")
print(f"MACD:     {daily_macd.upper()}")
print(f"Trend:    {daily_trend.upper()}")
print()

print("Structure de marche:")
print(f"   Trend: {daily_structure['structure']}")
print(f"   Bias: {daily_structure['bias']}")
print(f"   Range: {daily_structure['price_range_pct']:.2f}%")
print()

print("Niveaux cles daily:")
for lvl in daily_levels:
    emoji = 'SUP' if lvl['type'] == 'Support' else 'RES'
    print(f"   {emoji} {lvl['type']}: {lvl['level']:.5f} ({lvl['strength']}, {lvl['touches']} touches)")

# Analyse 1h
levels_1h = find_support_resistance(df_1h)
structure_1h = analyze_market_structure(df_1h)

print("\n" + "-" * 50)
print("1H - Structure de marche:")
print(f"   Trend: {structure_1h['structure']}")
print(f"   Bias: {structure_1h['bias']}")
print(f"   Range: {structure_1h['price_range_pct']:.2f}%")
print()

print("Niveaux cles 1H:")
for lvl in levels_1h:
    emoji = 'SUP' if lvl['type'] == 'Support' else 'RES'
    print(f"   {emoji} {lvl['type']}: {lvl['level']:.5f} ({lvl['strength']}, {lvl['touches']} touches)")

RSI:      69.2
MACD:     BEARISH
Trend:    DOWNTREND

Structure de marche:
   Trend: Uptrend (Higher Highs)
   Bias: Bullish
   Range: nan%

Niveaux cles daily:
   RES Resistance: 163.33850 (Medium, 2 touches)
   SUP Support: 160.95600 (Medium, 2 touches)

--------------------------------------------------
1H - Structure de marche:
   Trend: Consolidation (Ranging)
   Bias: Neutral
   Range: 3.66%

Niveaux cles 1H:
   SUP Support: 162.03986 (Strong, 14 touches)
   RES Resistance: 162.98364 (Strong, 11 touches)


In [6]:
# Donnees intraday
print("INTRADAY (5m)")
print("=" * 60)
df_5m = get_cached_data(SYMBOL + "=X", TF_INTRADAY["interval"], TF_INTRADAY["period"])

# Patterns, niveaux et structure
patterns_5m = detect_candlestick_patterns(df_5m)
levels_5m = find_support_resistance(df_5m)
structure_5m = analyze_market_structure(df_5m)

# Signal de base
signal = build_price_action_signal(df_5m, patterns_5m, levels_5m, structure_5m)

recent_patterns = [p for p in patterns_5m if p['index'] >= len(df_5m) - 5]
if recent_patterns:
    for p in recent_patterns:
        print(f"Pattern:  {p['pattern']} ({p['signal']}) - {p['strength']}")
else:
    print("Aucun pattern candlestick recent")
print()

INTRADAY (5m)
✓ Using cached data (age: 24m 46s)
Pattern:  Bearish Engulfing (Bearish Reversal) - Very Strong



In [7]:
# Enrichir le signal avec le contexte HTF
signal['pair'] = f'{SYMBOL} ({FULL_NAME})'
signal['daily_rsi'] = daily_rsi
signal['daily_macd'] = daily_macd
signal['daily_trend'] = daily_trend
signal['htf_daily_bias'] = daily_structure['bias']
signal['htf_daily_structure'] = daily_structure['structure']
signal['htf_1h_bias'] = structure_1h['bias']
signal['htf_1h_structure'] = structure_1h['structure']
signal['daily_support'] = [l['level'] for l in daily_levels if l['type'] == 'Support'][:2]
signal['daily_resistance'] = [l['level'] for l in daily_levels if l['type'] == 'Resistance'][:2]
signal['hourly_support'] = [l['level'] for l in levels_1h if l['type'] == 'Support'][:2]
signal['hourly_resistance'] = [l['level'] for l in levels_1h if l['type'] == 'Resistance'][:2]

print("\nSIGNAL COMBINE (Multi-Timeframe)")
print("=" * 70)
print(f"Daily bias:    {daily_structure['bias']:>8}  |  1H bias:      {structure_1h['bias']:>8}")
print(f"Intraday bias: {structure_5m['bias']:>8}  |  Prix: {signal['price']:.5f}")

# Verification concordance timeframes
biases = [daily_structure['bias'], structure_1h['bias'], structure_5m['bias']]
if all(b == 'Bullish' for b in biases):
    print("\nTOUS LES TIMEFRAMES SONT BULLISH - Signal haussier fort")
elif all(b == 'Bearish' for b in biases):
    print("\nTOUS LES TIMEFRAMES SONT BEARISH - Signal baissier fort")
elif daily_structure['bias'] == structure_1h['bias'] == 'Bullish' and structure_5m['bias'] != 'Bullish':
    print("\nHTF bullish mais intraday en contretendance - Attendre confirmation")
elif daily_structure['bias'] == structure_1h['bias'] == 'Bearish' and structure_5m['bias'] != 'Bearish':
    print("\nHTF bearish mais intraday en contretendance - Attendre confirmation")
else:
    print("\nTimeframes en desaccord - Prudence recommandee")


SIGNAL COMBINE (Multi-Timeframe)
Daily bias:     Bullish  |  1H bias:       Neutral
Intraday bias:  Bullish  |  Prix: 159.11099

Timeframes en desaccord - Prudence recommandee


In [8]:
# Analyse IA multi-timeframe
print("ANALYSE IA")
print("=" * 70)

ai_result = analyze_with_ai(signal)

if ai_result:
    emojis = {'BUY': '[BUY]', 'SELL': '[SELL]', 'HOLD': '[HOLD]'}
    ai_signal = ai_result.get('signal', 'UNKNOWN').upper()
    conf = ai_result.get('confidence', 0)

    print(f"\nSignal:       {ai_signal} {emojis.get(ai_signal, '')}")
    print(f"Confiance:    {conf}%")
    print(f"Raison:       {ai_result.get('reason', 'N/A')}")

    entry = ai_result.get('entry')
    sl = ai_result.get('stop_loss')
    tp = ai_result.get('take_profit')
    if entry is not None and sl is not None and tp is not None:
        print(f"Entree:       {entry:.5f}")
        print(f"Stop Loss:    {sl:.5f}")
        print(f"Take Profit:  {tp:.5f}")
else:
    print("Analyse IA indisponible")

ANALYSE IA

Signal:       HOLD [HOLD]
Confiance:    35%
Raison:       Conflicting signals: Short-term (NY) shows a Bearish Engulfing pattern and MACD is bearish, suggesting caution. However, the higher timeframe bias (HTF Daily/Structure) remains strongly bullish (Uptrend). The current price action is consolidating near key resistance levels (162.98 - 163.33), making an immediate directional trade risky.
